In [3]:
import os
import glob
import pandas as pd
import rasterio
import numpy as np

In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [6]:
!ls -l /content/drive/MyDrive/big_data_project/Data/soil

total 183679
-rw------- 1 root root  4073467 Feb 10 22:02 donnees_soil_1km.csv
-rw------- 1 root root  3984870 Feb 11 16:23 donnees_soil_1km_without_missing_valuesCollab.csv
-rw------- 1 root root  4074069 Feb 10 22:02 donnees_soil_1km_without_missing_values.csv
-rw------- 1 root root  3984268 Feb 11 16:23 donnees_soil_collab.csv
-rw------- 1 root root  4727424 Feb 10 23:39 HWSD2_clipped.tif
-rw------- 1 root root 72539068 Feb 11 13:02 HWSD2_LAYERSCollab_all.csv
-rw------- 1 root root   372841 Feb 11 10:31 HWSD2_LAYERS.csv
-rw------- 1 root root 91594752 Dec 16  2022 HWSD2.mdb
drwx------ 2 root root     4096 Feb 11 10:47 HWSD2_RASTER
-rw------- 1 root root  2725222 Feb 11 15:59 HWSD2_SMUCollab_all.csv
-rw------- 1 root root     5426 Feb 11 10:31 HWSD2_SMU.csv


In [7]:
df_soil = pd.read_csv("/content/drive/MyDrive/collab/f2/soil_filled.csv")
df_fire_grid = pd.read_csv("/content/drive/MyDrive/collab/f2/grid_with_fire.csv")

In [8]:
df_soil.shape

(608947, 25)

In [9]:
df_fire_grid.shape

(89198, 5)

In [10]:
df_fire_grid.columns

Index(['latitude', 'longitude', 'landcover', 'in_fire', 'dist_fire_km'], dtype='object')

In [11]:
df_fire_grid.drop(columns=['dist_fire_km'], inplace=True)
df_fire_grid.head(0)

,latitude,longitude,landcover,in_fire


In [12]:
from sklearn.neighbors import BallTree

# --- paramètres ---
seuil_km = 1.0
seuil_rad = seuil_km / 6371.0

# --- coords en radians (lat, lon) ---
soil_rad = np.radians(df_soil[["latitude", "longitude"]].values)
fire_rad = np.radians(df_fire_grid[["latitude", "longitude"]].values)

# --- arbre sur df_fire_grid ---
tree = BallTree(fire_rad, metric="haversine")

# --- nearest neighbor pour chaque point ---
dist, idx = tree.query(soil_rad, k=1)

# distance en km
dist_km = dist[:, 0] * 6371.0

# --- merge par index du plus proche ---
fire_cols = df_fire_grid.columns.difference(["longitude", "latitude"])
merged_soil = df_soil.copy()
merged_soil[fire_cols] = df_fire_grid.iloc[idx[:, 0]][fire_cols].values

# --- filtrer par seuil ---
merged_soil["dist_soil_km"] = dist_km
merged_soil["match_soil"] = (dist[:, 0] <= seuil_rad).astype(int)

# --- stats ---
print("Total all_points:", len(df_soil))
print("Match <= seuil:", merged_soil["match_soil"].sum())
print("No match:", (merged_soil["match_soil"] == 0).sum())
print("Distance min/max (km):", merged_soil["dist_soil_km"].min(), merged_soil["dist_soil_km"].max())

Total all_points: 608947
Match <= seuil: 608947
No match: 0
Distance min/max (km): 0.0 0.0


In [14]:
merged_soil.columns

Index(['latitude', 'longitude', 'landcover', 'SMU_ID', 'HWSD2_SMU_ID', 'LAYER',
       'COARSE', 'SAND', 'SILT', 'CLAY', 'TEXTURE_USDA', 'TEXTURE_SOTER',
       'BULK', 'ORG_CARBON', 'PH_WATER', 'CN_RATIO', 'CEC_SOIL', 'CEC_CLAY',
       'CEC_EFF', 'BSAT', 'ALUM_SAT', 'ESP', 'TCARBON_EQ', 'GYPSUM',
       'ELEC_COND', 'in_fire', 'dist_soil_km', 'match_soil'],
      dtype='object')

In [15]:
merged_soil.drop(columns=['dist_soil_km',"match_soil"], inplace=True)

In [16]:
merged_soil.columns

Index(['latitude', 'longitude', 'landcover', 'SMU_ID', 'HWSD2_SMU_ID', 'LAYER',
       'COARSE', 'SAND', 'SILT', 'CLAY', 'TEXTURE_USDA', 'TEXTURE_SOTER',
       'BULK', 'ORG_CARBON', 'PH_WATER', 'CN_RATIO', 'CEC_SOIL', 'CEC_CLAY',
       'CEC_EFF', 'BSAT', 'ALUM_SAT', 'ESP', 'TCARBON_EQ', 'GYPSUM',
       'ELEC_COND', 'in_fire'],
      dtype='object')

In [17]:
merged_soil.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 608947 entries, 0 to 608946
Data columns (total 26 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   latitude       608947 non-null  float64
 1   longitude      608947 non-null  float64
 2   landcover      608947 non-null  int64  
 3   SMU_ID         608947 non-null  int64  
 4   HWSD2_SMU_ID   608947 non-null  float64
 5   LAYER          608947 non-null  object 
 6   COARSE         608947 non-null  float64
 7   SAND           608947 non-null  float64
 8   SILT           608947 non-null  float64
 9   CLAY           608947 non-null  float64
 10  TEXTURE_USDA   608947 non-null  float64
 11  TEXTURE_SOTER  608947 non-null  object 
 12  BULK           608947 non-null  float64
 13  ORG_CARBON     608947 non-null  float64
 14  PH_WATER       608947 non-null  float64
 15  CN_RATIO       608947 non-null  float64
 16  CEC_SOIL       608947 non-null  float64
 17  CEC_CLAY       608947 non-nul

In [18]:
merged_soil["LAYER"].value_counts()

,count
LAYER,
D1,608947


In [19]:
merged_soil["TEXTURE_SOTER"].value_counts()

,count
TEXTURE_SOTER,
M,465963
-,54553
C,52582
F,35849


In [20]:
merged_soil.drop(columns=['LAYER'], inplace=True)

In [22]:
merged_soil.to_csv("/content/drive/MyDrive/collab/f2/soil_merged_cleaned_with_duplicate.csv", index=False)

In [ ]:


CHEMIN_FICHIER_SORTIE = '/content/drive/MyDrive/collab/f2/soil_merged_cleaned_without_duplicate.csv' 


COLONNES_GEO = ['latitude', 'longitude', 'in_fire'] 


COLONNES_TEXTURE = ['SAND', 'SILT', 'CLAY']


  
df = merged_soil.copy()  
lignes_originales = len(df)
    
print(f"Fichier chargé. Total des lignes (avec duplications de sol) : {lignes_originales}")
    
   
df['MAX_TEXTURE'] = df[COLONNES_TEXTURE].max(axis=1)

    
df_sorted = df.sort_values(
    by=COLONNES_GEO + ['MAX_TEXTURE'],
    ascending=[True, True, True, False]
)

   
df_cleaned = df_sorted.drop_duplicates(subset=COLONNES_GEO, keep='first').copy()
    
   
df_cleaned.drop(columns=['MAX_TEXTURE'], inplace=True)
    
  
df_cleaned.to_csv(CHEMIN_FICHIER_SORTIE, index=False)
    
lignes_supprimees = lignes_originales - len(df_cleaned)
print(f"\nNettoyage terminé. Total des lignes conservées (uniques par point) : {len(df_cleaned)}")
print(f"Nombre de lignes d'attributs de sol secondaires supprimées : {lignes_supprimees}")
print(f"Le fichier nettoyé est sauvegardé à : {CHEMIN_FICHIER_SORTIE}")
    
   

